In [1]:
#问题5：编写代码，对逻辑回归、决策树、k近邻和支持向量分类器，分别测试2个关键参数对预测精度的影响。

In [2]:
# 公共初始化代码｜读取训练集与固定五折，检查环境；不计算测试集指标
from pathlib import Path
from datetime import datetime
from time import perf_counter
import hashlib
import json
import sys
import uuid
import warnings
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from threadpoolctl import threadpool_limits
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import accuracy_score, f1_score

# 自动定位仓库中的作业目录，支持从仓库根目录或notebooks启动。
_location = Path.cwd()
for _parent in (_location, *_location.parents):
    _notebooks = _parent if _parent.name == "notebooks" else _parent / "notebooks"
    if (_notebooks / "imbalanced_classification" / "common.py").is_file():
        if str(_notebooks) not in sys.path:
            sys.path.insert(0, str(_notebooks))
        break
else:
    raise FileNotFoundError("未找到作业目录，请从仓库根目录或notebooks启动")
from imbalanced_classification.common import (
    PROJECT_DIR, check_environment, SOURCE_DIR, load_fixed_dataset, dataset_sha256
)
check_environment()
source = json.loads((SOURCE_DIR / "results/manifest.json").read_text(encoding="utf-8"))
assert sklearn.__version__ == source["versions"]["scikit-learn"], "sklearn版本与第一题不一致"
saved = load_fixed_dataset(SOURCE_DIR)
data_hash = dataset_sha256(saved["X"], saved["y"], saved["train_idx"], saved["test_idx"])
train_sample_ids = saved["train_idx"].copy()
X_train, y_train = saved["X"][train_sample_ids], saved["y"][train_sample_ids]
folds = [(saved[f"fold_{i}_fit"].copy(), saved[f"fold_{i}_validation"].copy()) for i in range(1, 6)]
assert data_hash == source["data_split_sha256"]
LABELS, MODEL_SEED = np.arange(10), source["model_seed"]
for fit_idx, val_idx in folds:
    assert not np.intersect1d(fit_idx, val_idx).size
    assert np.array_equal(np.sort(np.r_[fit_idx, val_idx]), np.arange(len(y_train)))
    assert np.array_equal(np.unique(y_train[fit_idx]), LABELS) and np.array_equal(np.unique(y_train[val_idx]), LABELS)
assert np.array_equal(np.sort(np.concatenate([v for _, v in folds])), np.arange(len(y_train)))
Q5_DIR = PROJECT_DIR / "q5_runs" / (datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6])
Q5_DIR.mkdir(parents=True, exist_ok=False)
(Q5_DIR / "validation_predictions").mkdir()
print("问题5 v1.0；Python", sys.version.split()[0], "；sklearn", sklearn.__version__)
print("仅使用训练集：", X_train.shape, "；固定5折；模型种子：", MODEL_SEED)
print("本题输出目录：", Q5_DIR)

问题5 v1.0；Python 3.12.13 ；sklearn 1.9.0
仅使用训练集： (3600, 24) ；固定5折；模型种子： 20261022
本题输出目录： D:\Python for Data Analysis\notebooks\imbalanced_classification\q5_runs\20261021_124751_b53362


In [3]:
# 固定模型、预处理、参数范围与锚点
base_models = {
    "LogisticRegression": LogisticRegression(solver="saga", max_iter=2000, random_state=MODEL_SEED),
    "DecisionTree": DecisionTreeClassifier(random_state=MODEL_SEED),
    "KNN": KNeighborsClassifier(),
    "SVC": SVC(random_state=MODEL_SEED),
}
base_pipelines = {name: Pipeline([
    ("scale", "passthrough" if name == "DecisionTree" else StandardScaler()),
    ("model", model),
]) for name, model in base_models.items()}
SCANS = [
    ("lr_C", "LogisticRegression", "C", [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]),
    ("lr_l1_ratio", "LogisticRegression", "l1_ratio", [0.0, 0.25, 0.5, 0.75, 1.0]),
    ("tree_max_depth", "DecisionTree", "max_depth", [1, 2, 4, 6, 10, 16, None]),
    ("tree_min_samples_leaf", "DecisionTree", "min_samples_leaf", [1, 2, 5, 10, 20, 50]),
    ("knn_n_neighbors", "KNN", "n_neighbors", [1, 3, 5, 9, 15, 31, 61]),
    ("knn_p", "KNN", "p", [1, 1.5, 2, 3]),
    ("svc_C", "SVC", "C", [0.01, 0.1, 1.0, 10.0, 100.0]),
    ("svc_gamma", "SVC", "gamma", [0.001, 0.005, 0.01, 1/24, 0.1, 0.5, 1.0]),
]
selection_rules = {"C": "smaller", "gamma": "smaller", "max_depth": "shallower_None_last",
    "min_samples_leaf": "larger", "n_neighbors": "larger", "l1_ratio": "larger",
    "p": "priority_2_1_1.5_3"}
plan = dict(scans=SCANS, base_parameters={k: v.get_params(deep=False) for k, v in base_models.items()},
    preprocessing={k: "none" if k == "DecisionTree" else "StandardScaler" for k in base_models},
    selection_metric="validation_macro_f1_mean", tie_atol=1e-12, tie_rules=selection_rules)
(Q5_DIR / "q5_plan.json").write_text(json.dumps(plan, ensure_ascii=False, indent=2), encoding="utf-8")
print("候选总数：", sum(len(values) for _, _, _, values in SCANS), "；总拟合次数：", 5*sum(len(values) for _, _, _, values in SCANS))
display(pd.DataFrame(SCANS, columns=["scan_id", "model", "parameter", "values"]))
print("范围已预定并保存；每个扫描从base_pipelines重新克隆，不继承前一次扫描的推荐参数。")

候选总数： 47 ；总拟合次数： 235


,scan_id,model,parameter,values
0,lr_C,LogisticRegression,C,"[0.001, 0.01, 0.1, 1.0, 10.0, 100.0]"
1,lr_l1_ratio,LogisticRegression,l1_ratio,"[0.0, 0.25, 0.5, 0.75, 1.0]"
2,tree_max_depth,DecisionTree,max_depth,"[1, 2, 4, 6, 10, 16, None]"
3,tree_min_samples_leaf,DecisionTree,min_samples_leaf,"[1, 2, 5, 10, 20, 50]"
4,knn_n_neighbors,KNN,n_neighbors,"[1, 3, 5, 9, 15, 31, 61]"
5,knn_p,KNN,p,"[1, 1.5, 2, 3]"
6,svc_C,SVC,C,"[0.01, 0.1, 1.0, 10.0, 100.0]"
7,svc_gamma,SVC,gamma,"[0.001, 0.005, 0.01, 0.041666666666666664, 0.1..."


范围已预定并保存；每个扫描从base_pipelines重新克隆，不继承前一次扫描的推荐参数。


In [4]:
# 单候选五折验证函数｜定义本身不训练
fold_rows, warning_rows = [], []
def evaluate_candidate(scan_id, name, parameter, value, candidate_index):
    candidate = clone(base_pipelines[name]).set_params(**{f"model__{parameter}": value})
    oof_parts = []
    for fold, (fit_idx, val_idx) in enumerate(folds, start=1):
        model = clone(candidate)  # 每折全新Pipeline，缩放仅在该折训练部分拟合。
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            start = perf_counter()
            model.fit(X_train[fit_idx], y_train[fit_idx])
            fit_seconds = perf_counter() - start
            pred_val = model.predict(X_train[val_idx])
            pred_train = model.predict(X_train[fit_idx])
        estimator = model.named_steps["model"]
        convergence_warning = any(issubclass(w.category, ConvergenceWarning) for w in caught)
        fit_status = int(getattr(estimator, "fit_status_", 0))
        for item in caught:
            warning_rows.append(dict(scan_id=scan_id, candidate_index=candidate_index, fold=fold,
                category=item.category.__name__, message=str(item.message)))
        fold_rows.append(dict(scan_id=scan_id, model=name, parameter=parameter,
            candidate_index=candidate_index, value_label=str(value), fold=fold,
            train_accuracy=accuracy_score(y_train[fit_idx], pred_train),
            train_macro_f1=f1_score(y_train[fit_idx], pred_train, labels=LABELS, average="macro", zero_division=0),
            validation_accuracy=accuracy_score(y_train[val_idx], pred_val),
            validation_macro_f1=f1_score(y_train[val_idx], pred_val, labels=LABELS, average="macro", zero_division=0),
            convergence_warning=convergence_warning, fit_status=fit_status, fit_seconds=fit_seconds,
            max_n_iter=int(np.max(estimator.n_iter_)) if hasattr(estimator, "n_iter_") else None,
            nonzero_coefficients=int(np.count_nonzero(estimator.coef_)) if hasattr(estimator, "coef_") else None,
            tree_depth=estimator.get_depth() if hasattr(estimator, "get_depth") else None,
            support_vectors=int(np.sum(estimator.n_support_)) if hasattr(estimator, "n_support_") else None))
        oof_parts.append(pd.DataFrame(dict(fold=fold, sample_id=train_sample_ids[val_idx],
                                           y_true=y_train[val_idx], y_pred=pred_val)))
    # 每个候选完成后保存折外预测及累计折得分，不必等全部候选结束。
    pd.concat(oof_parts, ignore_index=True).to_csv(
        Q5_DIR / "validation_predictions" / f"{scan_id}_{candidate_index:02d}.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(fold_rows).to_csv(Q5_DIR / "q5_fold_scores.csv", index=False, encoding="utf-8-sig")
    (Q5_DIR / "q5_warnings.json").write_text(json.dumps(warning_rows, ensure_ascii=False, indent=2), encoding="utf-8")

In [5]:
# 运行8组单参数扫描｜输出：进度、验证均值与警告
fold_rows.clear(); warning_rows.clear()  # 重新运行本单元会重做全部235次拟合。
total = sum(len(values) for _, _, _, values in SCANS)
done = 0
with threadpool_limits(limits=1):
    for scan_id, name, parameter, values in SCANS:
        for candidate_index, value in enumerate(values):
            evaluate_candidate(scan_id, name, parameter, value, candidate_index)
            done += 1
            recent = fold_rows[-len(folds):]
            mean_f1 = np.mean([row["validation_macro_f1"] for row in recent])
            bad_folds = sum(row["convergence_warning"] or row["fit_status"] != 0 for row in recent)
            print(f"[{done}/{total}] {name} {parameter}={value}：验证macro-F1={mean_f1:.6f}；求解异常折数={bad_folds}", flush=True)
fold_scores = pd.DataFrame(fold_rows)
assert len(fold_scores) == total * len(folds)
print("全部候选完成；警告总数：", len(warning_rows))
for message in sorted({w["message"] for w in warning_rows}):
    print("警告全文：", message)

[1/47] LogisticRegression C=0.001：验证macro-F1=0.173218；求解异常折数=0
[2/47] LogisticRegression C=0.01：验证macro-F1=0.306539；求解异常折数=0
[3/47] LogisticRegression C=0.1：验证macro-F1=0.374704；求解异常折数=0
[4/47] LogisticRegression C=1.0：验证macro-F1=0.386444；求解异常折数=0
[5/47] LogisticRegression C=10.0：验证macro-F1=0.385530；求解异常折数=0
[6/47] LogisticRegression C=100.0：验证macro-F1=0.385407；求解异常折数=0
[7/47] LogisticRegression l1_ratio=0.0：验证macro-F1=0.386444；求解异常折数=0
[8/47] LogisticRegression l1_ratio=0.25：验证macro-F1=0.385947；求解异常折数=0
[9/47] LogisticRegression l1_ratio=0.5：验证macro-F1=0.385816；求解异常折数=0
[10/47] LogisticRegression l1_ratio=0.75：验证macro-F1=0.383552；求解异常折数=0
[11/47] LogisticRegression l1_ratio=1.0：验证macro-F1=0.384080；求解异常折数=0
[12/47] DecisionTree max_depth=1：验证macro-F1=0.093792；求解异常折数=0
[13/47] DecisionTree max_depth=2：验证macro-F1=0.108108；求解异常折数=0
[14/47] DecisionTree max_depth=4：验证macro-F1=0.205710；求解异常折数=0
[15/47] DecisionTree max_depth=6：验证macro-F1=0.293030；求解异常折数=0
[16/47] DecisionTree max_depth=10：验证

In [6]:
# 汇总与预定规则选参｜不使用测试集，不拼接两个最优参数
summary_rows, selections = [], []
score_columns = ["train_accuracy", "train_macro_f1", "validation_accuracy", "validation_macro_f1"]
def simplicity_key(parameter, value):
    if parameter in ("C", "gamma"): return value
    if parameter == "max_depth": return float("inf") if value is None else value
    if parameter in ("min_samples_leaf", "n_neighbors", "l1_ratio"): return -value
    return [2, 1, 1.5, 3].index(value)  # p没有统一复杂度排序，按预设默认优先规则。

for scan_id, name, parameter, values in SCANS:
    for index, value in enumerate(values):
        part = fold_scores.query("scan_id == @scan_id and candidate_index == @index")
        row = dict(scan_id=scan_id, model=name, parameter=parameter, candidate_index=index,
                   value_label=str(value), fold_count=len(part),
                   eligible=len(part)==5 and not part.convergence_warning.any() and part.fit_status.eq(0).all())
        for metric in score_columns:
            row[f"{metric}_mean"], row[f"{metric}_std"] = part[metric].mean(), part[metric].std(ddof=1)
        row["generalization_gap"] = row["train_macro_f1_mean"] - row["validation_macro_f1_mean"]
        summary_rows.append(row)
    candidates = [row for row in summary_rows if row["scan_id"] == scan_id and row["eligible"]]
    if not candidates:
        selections.append(dict(scan_id=scan_id, model=name, parameter=parameter, status="no_eligible_candidate"))
        continue
    best_score = max(row["validation_macro_f1_mean"] for row in candidates)
    tied = [row for row in candidates if abs(row["validation_macro_f1_mean"] - best_score) <= 1e-12]
    best = min(tied, key=lambda row: simplicity_key(parameter, values[row["candidate_index"]]))
    selections.append(dict(scan_id=scan_id, model=name, parameter=parameter, status="selected",
        selected_value=values[best["candidate_index"]], candidate_index=best["candidate_index"],
        validation_macro_f1_mean=best["validation_macro_f1_mean"], validation_macro_f1_std=best["validation_macro_f1_std"],
        validation_accuracy_mean=best["validation_accuracy_mean"], tie_count=len(tied)))
summary = pd.DataFrame(summary_rows)
for scan_id, _, _, _ in SCANS:
    print("扫描：", scan_id)
    display(summary.query("scan_id == @scan_id")[["value_label", "train_macro_f1_mean",
        "validation_macro_f1_mean", "validation_macro_f1_std", "validation_accuracy_mean",
        "validation_accuracy_std", "generalization_gap", "eligible"]].round(6))
print("按预定规则推荐：每行只对应一个扫描，不能将两行参数直接拼成联合最优。")
display(pd.DataFrame(selections).round(6))

扫描： lr_C


,value_label,train_macro_f1_mean,validation_macro_f1_mean,validation_macro_f1_std,validation_accuracy_mean,validation_accuracy_std,generalization_gap,eligible
0,0.001,0.174864,0.173218,0.008038,0.466667,0.008039,0.001646,True
1,0.01,0.322926,0.306539,0.027508,0.547222,0.015528,0.016387,True
2,0.1,0.394211,0.374704,0.023944,0.567500,0.011436,0.019507,True
3,1.0,0.405992,0.386444,0.029220,0.568056,0.011579,0.019548,True
4,10.0,0.407806,0.385530,0.028886,0.567222,0.011094,0.022276,True
5,100.0,0.407706,0.385407,0.028697,0.567222,0.011094,0.022299,True


扫描： lr_l1_ratio


,value_label,train_macro_f1_mean,validation_macro_f1_mean,validation_macro_f1_std,validation_accuracy_mean,validation_accuracy_std,generalization_gap,eligible
6,0.0,0.405992,0.386444,0.029220,0.568056,0.011579,0.019548,True
7,0.25,0.405273,0.385947,0.029230,0.567778,0.011128,0.019325,True
8,0.5,0.404917,0.385816,0.028462,0.568333,0.011554,0.019102,True
9,0.75,0.404486,0.383552,0.027193,0.568333,0.010821,0.020934,True
10,1.0,0.403769,0.384080,0.028334,0.568611,0.010695,0.019689,True


扫描： tree_max_depth


,value_label,train_macro_f1_mean,validation_macro_f1_mean,validation_macro_f1_std,validation_accuracy_mean,validation_accuracy_std,generalization_gap,eligible
11,1,0.096252,0.093792,0.003629,0.361111,0.011495,0.002460,True
12,2,0.116823,0.108108,0.021873,0.356944,0.013249,0.008715,True
13,4,0.229380,0.205710,0.006203,0.443611,0.017280,0.023670,True
14,6,0.384329,0.293030,0.019516,0.501389,0.013784,0.091299,True
15,10,0.801695,0.368765,0.023331,0.528056,0.017878,0.432930,True
16,16,0.987929,0.372677,0.016771,0.514722,0.004969,0.615252,True
17,None,1.000000,0.367739,0.017982,0.508333,0.012028,0.632261,True


... (8 outputs left out)

扫描： svc_gamma


,value_label,train_macro_f1_mean,validation_macro_f1_mean,validation_macro_f1_std,validation_accuracy_mean,validation_accuracy_std,generalization_gap,eligible
40,0.001,0.223734,0.218057,0.006144,0.520556,0.010731,0.005677,True
41,0.005,0.385262,0.365556,0.020875,0.619722,0.010130,0.019706,True
42,0.01,0.500345,0.442399,0.027432,0.673889,0.012321,0.057946,True
43,0.041666666666666664,0.815485,0.645576,0.037860,0.787778,0.014646,0.169909,True
44,0.1,0.947990,0.623593,0.033026,0.781389,0.008252,0.324397,True
45,0.5,1.000000,0.049518,0.001513,0.317222,0.001584,0.950482,True
46,1.0,1.000000,0.048005,0.000088,0.315833,0.000761,0.951995,True


按预定规则推荐：每行只对应一个扫描，不能将两行参数直接拼成联合最优。


,scan_id,model,parameter,status,selected_value,candidate_index,validation_macro_f1_mean,validation_macro_f1_std,validation_accuracy_mean,tie_count
0,lr_C,LogisticRegression,C,selected,1.000000,3,0.386444,0.029220,0.568056,1
1,lr_l1_ratio,LogisticRegression,l1_ratio,selected,0.000000,0,0.386444,0.029220,0.568056,1
2,tree_max_depth,DecisionTree,max_depth,selected,16.000000,5,0.372677,0.016771,0.514722,1
3,tree_min_samples_leaf,DecisionTree,min_samples_leaf,selected,1.000000,0,0.367739,0.017982,0.508333,1
4,knn_n_neighbors,KNN,n_neighbors,selected,1.000000,0,0.580462,0.046502,0.703056,1
5,knn_p,KNN,p,selected,1.500000,1,0.553715,0.038524,0.720556,1
6,svc_C,SVC,C,selected,10.000000,3,0.697637,0.030924,0.800556,1
7,svc_gamma,SVC,gamma,selected,0.041667,3,0.645576,0.037860,0.787778,1


In [7]:
# 辅助代码：折内赢家与保存｜用于判断推荐的稳定性，不用于额外挑选参数
fold_winners = []
for scan_id, name, parameter, values in SCANS:
    valid_indices = summary.query("scan_id == @scan_id and eligible").candidate_index.tolist()
    for fold in range(1, 6):
        part = fold_scores.query("scan_id == @scan_id and fold == @fold").copy()
        part = part[part.candidate_index.isin(valid_indices)]
        if part.empty: continue
        best_score = part.validation_macro_f1.max()
        tied = part.loc[(part.validation_macro_f1 - best_score).abs() <= 1e-12, "candidate_index"].tolist()
        winner = min(tied, key=lambda index: simplicity_key(parameter, values[index]))
        fold_winners.append(dict(scan_id=scan_id, fold=fold, candidate_index=winner, value_label=str(values[winner])))
winner_table = pd.DataFrame(fold_winners, columns=["scan_id", "fold", "candidate_index", "value_label"])
summary.to_csv(Q5_DIR / "q5_summary.csv", index=False, encoding="utf-8-sig")
winner_table.to_csv(Q5_DIR / "q5_fold_winners.csv", index=False, encoding="utf-8-sig")
(Q5_DIR / "q5_selections.json").write_text(json.dumps(selections, ensure_ascii=False, indent=2), encoding="utf-8")
manifest = dict(code_version="1.0", source_dir=str(SOURCE_DIR), data_split_sha256=data_hash,
    python=sys.version, sklearn=sklearn.__version__, executable=sys.executable,
    model_seed=MODEL_SEED, cv_seed=source["cv_seed"], folds=5, training_samples=len(y_train),
    labels=LABELS.tolist(), zero_division=0, std_definition="sample std ddof=1; not a confidence interval",
    source_fold_indices="saved folds from question 1; positions within training set",
    selection_metric="mean of 5 validation macro-F1 scores", test_set_evaluated=False,
    total_candidates=total, total_fits=len(fold_scores), warning_count=len(warning_rows),
    created_at=datetime.now().astimezone().isoformat(), output_dir=str(Q5_DIR))
(Q5_DIR / "q5_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("折内最优值出现次数（五折不是五次独立实验，次数仅作稳定性描述）：")
display(winner_table.groupby(["scan_id", "value_label"]).size().rename("fold_wins").reset_index())
print("保存完成：", Q5_DIR)
print("测试集未用于评分或选参；请发送核心代码、各扫描得分表、推荐表和警告（如有）。")

折内最优值出现次数（五折不是五次独立实验，次数仅作稳定性描述）：


,scan_id,value_label,fold_wins
0,knn_n_neighbors,1,5
1,knn_p,1,1
2,knn_p,1.5,2
3,knn_p,3,2
4,lr_C,1.0,4
5,lr_C,10.0,1
6,lr_l1_ratio,0.0,2
7,lr_l1_ratio,0.25,1
8,lr_l1_ratio,0.5,1
9,lr_l1_ratio,1.0,1


保存完成： D:\Python for Data Analysis\notebooks\imbalanced_classification\q5_runs\20261021_124751_b53362
测试集未用于评分或选参；请发送核心代码、各扫描得分表、推荐表和警告（如有）。
